In [ ]:
import os
import numpy as np
import time
import torch
import chess
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import DataLoader, Dataset 
from chess import Board, pgn 
from tqdm import tqdm
import pickle

# Data Preprocessing

In [ ]:
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
    return games
files = [file for file in os.listdir("../data/pgn") if file.endswith(".pgn")]
LIMITS = min(len(files), 28)
games = []
i = 1
for file in tqdm(files):
    games.extend(load_pgn(f"../data/pgn/{file}"))
    if i >= LIMITS:
        break
    i += 1

 34%|███▍      | 27/79 [01:28<02:49,  3.27s/it]


In [8]:
def board_to_mat(board: chess.Board):
    matrix = np.zeros((13, 8, 8), dtype=np.float32)
    for square, piece in board.piece_map().items():
        row = 7 - (square // 8)
        col = square % 8
        piece_plane = (piece.piece_type - 1) + (0 if piece.color else 6)
        matrix[piece_plane, row, col] = 1
    for move in board.legal_moves:
        r = 7 - (move.to_square // 8)
        c = move.to_square % 8
        matrix[12, r, c] = 1
    return matrix

def create_input_for_nn(games):
    x, y = [], []
    for game in games:
        board = game.board().copy()
        for move in game.mainline_moves():
            x.append(board_to_mat(board))
            y.append(move.uci())
            board.push(move)
    return np.array(x, dtype=np.float32), y


def encode_moves(moves):
    unique_moves = list(set(moves))
    move_to_int = {m: i for i, m in enumerate(unique_moves)}
    encoded = np.array([move_to_int[m] for m in moves], dtype=np.int32)
    return encoded, move_to_int

In [9]:
x, y = create_input_for_nn(games)
y, move_to_int = encode_moves(y)
num_classes = len(move_to_int)
x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

KeyboardInterrupt: 

In [ ]:
class ChessDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

class ChessModel(nn.Module):
    def __init__(self, num_classes):
        super(ChessModel, self).__init__()
        self.conv1 = nn.Conv2d(13, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(8 * 8 * 128, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.relu = nn.ReLU()
        nn.init.kaiming_uniform_(self.conv1.weight, nonlinearity='relu')
        nn.init.kaiming_uniform_(self.conv2.weight, nonlinearity='relu')
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
dataset = ChessDataset(x, y)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')
model = ChessModel(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training

In [ ]:
num_epochs = 100
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)  
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(f'Epoch {epoch + 1 + 50}/{num_epochs + 1 + 50}, Loss: {running_loss / len(dataloader):.4f}, Time: {minutes}m{seconds}s')

torch.save(model.state_dict(), "../models/TORCH_100EPOCHS.pth")
with open("../models/heavy_move_to_int", "wb") as file:
    pickle.dump(move_to_int, file)